In [1]:
import numpy as np
import pandas as pd
import ast
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder
from kmodes.kmodes import KModes
from scipy import stats

warnings.filterwarnings("ignore")

## **Featurization**

In [2]:
CHUNK_SIZE = 2_000_000

def regression_slope(x_values, y_values) -> float:
    x = np.asarray(x_values, dtype="float64")
    y = np.asarray(y_values, dtype="float64")
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 2 or np.unique(x).size < 2:
        return np.nan
    return float(np.polyfit(x, y, 1)[0])


def slope_from_datetime(dt_series: pd.Series, value_series: pd.Series) -> float:
    dt = pd.to_datetime(dt_series, errors="coerce", utc=True)
    if dt.notna().sum() < 2:
        return np.nan
    x_days = (dt - dt.min()).dt.total_seconds() / 86400.0
    return regression_slope(x_days.to_numpy(), pd.to_numeric(value_series, errors="coerce").to_numpy())


def parse_message_list(raw):
    if pd.isna(raw):
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass
    return []

In [3]:
students = pd.read_csv("out/students.csv", usecols=["user_id"]).drop_duplicates()

# Base pageview table used by multiple features.
pageviews = pd.read_csv(
    "out/pageviews.csv",
    usecols=["id", "user_id", "created_at"],
    low_memory=False,
)
pageviews["created_at"] = pd.to_datetime(pageviews["created_at"], errors="coerce", utc=True)
pageviews = pageviews.dropna(subset=["user_id", "created_at"])
pageviews["user_id"] = pageviews["user_id"].astype("int64")
pageviews["activity_day"] = pageviews["created_at"].dt.floor("D")

In [4]:
# 1) Active Days & Spacing Effect
pv_sorted = pageviews.sort_values(["user_id", "created_at"]).copy()
pv_sorted["login_gap_days"] = (
    pv_sorted.groupby("user_id")["created_at"].diff().dt.total_seconds() / 86400.0
)

active_days_spacing = (
    pv_sorted.groupby("user_id")
    .agg(
        active_days=("activity_day", "nunique"),
        login_events=("id", "count"),
        login_gap_days_mean=("login_gap_days", "mean"),
        login_gap_days_var=("login_gap_days", "var"),
    )
    .reset_index()
)

In [5]:
# 2) Total Active Learning Time + 12) Engagement Trend (from events/0 heartbeat)
IDLE_THRESHOLD_MS = 60_000
MAX_HEARTBEAT_S = 300

pageview_to_user = pageviews.set_index("id")["user_id"]
active_time_parts = []
daily_active_parts = []

for chunk in pd.read_csv(
    "out/events/0.csv",
    usecols=["timestamp", "lastActivity", "pageview_id"],
    chunksize=CHUNK_SIZE,
):
    chunk = chunk.dropna(subset=["pageview_id"])
    chunk["user_id"] = chunk["pageview_id"].map(pageview_to_user)
    chunk = chunk.dropna(subset=["user_id"])
    if chunk.empty:
        continue

    chunk["user_id"] = chunk["user_id"].astype("int64")
    chunk["ts_ms"]   = pd.to_numeric(chunk["timestamp"], errors="coerce")
    chunk["last_ms"] = pd.to_numeric(chunk["lastActivity"], errors="coerce")

    # Sort by pageview then time so diffs are meaningful within each session.
    chunk = chunk.sort_values(["pageview_id", "ts_ms"])
    
    # How long this heartbeat "covers"
    chunk["interval_ms"] = chunk.groupby("pageview_id")["ts_ms"].diff().fillna(0).clip(lower=0)

    # How long since user last interacted
    idle_ms = (chunk["ts_ms"] - chunk["last_ms"]).clip(lower=0).fillna(IDLE_THRESHOLD_MS + 1)

    # Only count the interval if the user was engaged within the last 60 s
    chunk["active_seconds"] = np.where(
    idle_ms <= IDLE_THRESHOLD_MS,
    (chunk["interval_ms"] / 1000.0).clip(upper=MAX_HEARTBEAT_S), 0.0,)

    chunk["event_day"] = (
        pd.to_datetime(chunk["ts_ms"], unit="ms", errors="coerce", utc=True)
        .dt.floor("D")
    )
    chunk = chunk.dropna(subset=["event_day"])

    active_time_parts.append(
        chunk.groupby("user_id", as_index=False)["active_seconds"].sum()
    )
    daily_active_parts.append(
        chunk.groupby(["user_id", "event_day"], as_index=False)["active_seconds"].sum()
    )

if active_time_parts:
    total_active_learning = (
        pd.concat(active_time_parts, ignore_index=True)
        .groupby("user_id", as_index=False)["active_seconds"]
        .sum()
        .rename(columns={"active_seconds": "total_active_learning_time_seconds"})
    )
else:
    total_active_learning = pd.DataFrame(columns=["user_id", "total_active_learning_time_seconds"])

if daily_active_parts:
    daily_active = (
        pd.concat(daily_active_parts, ignore_index=True)
        .groupby(["user_id", "event_day"], as_index=False)["active_seconds"]
        .sum()
    )
    engagement_trend = (
        daily_active.groupby("user_id")
        .apply(lambda g: slope_from_datetime(g["event_day"], g["active_seconds"]))
        .rename("engagement_trend")
        .reset_index()
    )
else:
    engagement_trend = pd.DataFrame(columns=["user_id", "engagement_trend"])

In [6]:
# 3) Video Completion Rate + pauses/replays
video = pd.read_csv(
    "out/events/m.csv",
    usecols=["timestamp", "eventType", "currentTime", "duration", "mediaId", "pageview_id"],
)
video["user_id"] = video["pageview_id"].map(pageview_to_user)
video = video.dropna(subset=["user_id", "duration"])
video["user_id"] = video["user_id"].astype("int64")
video["duration"] = pd.to_numeric(video["duration"], errors="coerce")
video["currentTime"] = pd.to_numeric(video["currentTime"], errors="coerce")
video = video[video["duration"] > 0]
video["media_key"] = video["mediaId"].astype("string").fillna("unknown")
video["completion_ratio"] = (video["currentTime"] / video["duration"]).clip(lower=0, upper=1)

video_keys = ["user_id", "pageview_id", "media_key"]
video_completion = (
    video.groupby(video_keys, as_index=False)["completion_ratio"].max()
)
video_features = (
    video_completion.groupby("user_id")
    .agg(
        avg_video_completion_ratio=("completion_ratio", "mean"),
        video_completion_rate=("completion_ratio", lambda s: (s >= 0.9).mean()),
        videos_interacted=("completion_ratio", "size"),
    )
    .reset_index()
)

pause_counts = (
    video[video["eventType"].astype("string").str.lower() == "pause"]
    .groupby("user_id")
    .size()
    .rename("pause_events")
)

video = video.copy()
video["timestamp_ms"] = pd.to_numeric(video["timestamp"], errors="coerce")
video = video.sort_values(video_keys + ["timestamp_ms"])
video["delta_current"] = video.groupby(video_keys)["currentTime"].diff()
replay_counts = (
    (video["delta_current"] < -1)
    .groupby(video["user_id"])
    .sum()
    .rename("replay_events")
)

video_features = video_features.merge(pause_counts, on="user_id", how="left")
video_features = video_features.merge(replay_counts, on="user_id", how="left")
video_features[["pause_events", "replay_events"]] = video_features[["pause_events", "replay_events"]].fillna(0)
video_features["pause_per_video"] = video_features["pause_events"] / video_features["videos_interacted"].replace(0, np.nan)
video_features["replay_per_video"] = video_features["replay_events"] / video_features["videos_interacted"].replace(0, np.nan)

In [7]:
# 4) Scroll Depth
# Load scroll position from heartbeats (most representative sampling)
heartbeats = pd.read_csv(
    "out/events/0.csv",
    usecols=["pageview_id", "scrollY"]
)
heartbeats["scrollY"] = pd.to_numeric(heartbeats["scrollY"], errors="coerce")

# Deepest scroll position reached per pageview
max_scroll = heartbeats.groupby("pageview_id")["scrollY"].max().reset_index()

# Load page dimensions from s.csv
dims = pd.read_csv(
    "out/events/s.csv",
    usecols=["pageview_id", "windowHeight", "fullHeight"]
)
dims["windowHeight"] = pd.to_numeric(dims["windowHeight"], errors="coerce")
dims["fullHeight"]   = pd.to_numeric(dims["fullHeight"],   errors="coerce")

# Take the most common (mode) dimensions per pageview to avoid outliers from resize events
dims = (
    dims.dropna(subset=["windowHeight", "fullHeight"])
    .query("fullHeight > 0")
    .groupby("pageview_id")[["windowHeight", "fullHeight"]]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)

# Join scroll position with page dimensions
scroll = dims.merge(max_scroll, on="pageview_id", how="inner")
scroll = scroll.dropna(subset=["scrollY"])

# Map to user
scroll["user_id"] = scroll["pageview_id"].map(pageview_to_user)
scroll = scroll.dropna(subset=["user_id"])
scroll["user_id"] = scroll["user_id"].astype("int64")

# Scroll depth: how far down the page the user reached
scroll["scroll_depth_ratio"] = (
    (scroll["scrollY"] + scroll["windowHeight"]) / scroll["fullHeight"]
).clip(lower=0, upper=1)

scroll_features = (
    scroll.groupby("user_id")
    .agg(
        avg_scroll_depth_ratio=("scroll_depth_ratio", "mean"),
        p90_scroll_depth_ratio=("scroll_depth_ratio", lambda s: s.quantile(0.9)),
    )
    .reset_index()
)

In [8]:
# 5) Chatbot Session Depth & Prompt Effort
gymi = pd.read_csv("out/gymitrainer.csv", usecols=["user_id", "content"])
thread_rows = []
for user_id, raw_content in gymi[["user_id", "content"]].itertuples(index=False):
    msgs = parse_message_list(raw_content)
    user_lengths = []
    user_count = 0
    bot_count = 0
    for msg in msgs:
        if not isinstance(msg, dict):
            continue
        text = str(msg.get("text", "")).strip()
        if msg.get("gymitrainer") is True:
            bot_count += 1
        else:
            user_count += 1
            user_lengths.append(len(text))

    if user_count == 0 and bot_count == 0:
        continue

    thread_rows.append(
        {
            "user_id": int(user_id),
            "session_depth_turns": min(user_count, bot_count),
            "user_messages": user_count,
            "avg_prompt_chars_in_thread": float(np.mean(user_lengths)) if user_lengths else np.nan,
        }
    )

if thread_rows:
    chat_df = pd.DataFrame(thread_rows)
    chat_features = (
        chat_df.groupby("user_id")
        .agg(
            chatbot_session_depth=("session_depth_turns", "mean"),
            chatbot_user_messages_per_thread=("user_messages", "mean"),
            chatbot_prompt_effort_chars=("avg_prompt_chars_in_thread", "mean"),
        )
        .reset_index()
    )
else:
    chat_features = pd.DataFrame(
        columns=[
            "user_id",
            "chatbot_session_depth",
            "chatbot_user_messages_per_thread",
            "chatbot_prompt_effort_chars",
        ]
    )

In [9]:
# 6) Question Review Rate
q_events = pd.read_csv("out/events/q.csv", usecols=["pageview_id", "questionNumber"])
q_events["user_id"] = q_events["pageview_id"].map(pageview_to_user)
q_events = q_events.dropna(subset=["user_id", "questionNumber"])
q_events["user_id"] = q_events["user_id"].astype("int64")

question_views = (
    q_events.groupby(["user_id", "questionNumber"], as_index=False)
    .size()
    .rename(columns={"size": "view_count"})
)
question_views["repeat_views"] = (question_views["view_count"] - 1).clip(lower=0)

question_review = (
    question_views.groupby("user_id")
    .agg(
        total_question_views=("view_count", "sum"),
        unique_questions_viewed=("questionNumber", "nunique"),
        repeated_question_views=("repeat_views", "sum"),
    )
    .reset_index()
)
question_review["question_review_rate"] = question_review["repeated_question_views"] / question_review["total_question_views"].replace(0, np.nan)

In [10]:
# 7) Average Score Ratio and Score Variance
quiz_results = pd.read_csv(
    "out/quiz_results.csv",
    usecols=["user_id", "quiz_id", "question_id", "points", "max_points", "time", "hint_count", "correct"],
)
quiz_results["score_ratio"] = (
    pd.to_numeric(quiz_results["points"], errors="coerce")
    / pd.to_numeric(quiz_results["max_points"], errors="coerce").replace(0, np.nan)
)

math_results = pd.read_csv(
    "out/math_results.csv",
    usecols=["user_id", "question_id", "points", "max_points", "timestamp"],
)
math_results["score_ratio"] = (
    pd.to_numeric(math_results["points"], errors="coerce")
    / pd.to_numeric(math_results["max_points"], errors="coerce").replace(0, np.nan)
)
math_results["attempt_time"] = pd.to_datetime(math_results["timestamp"], unit="s", errors="coerce", utc=True)
math_results["question_key"] = "math_" + math_results["question_id"].astype("string")

quiz_attempts = quiz_results.copy()
quiz_attempts["attempt_time"] = pd.to_datetime(quiz_attempts["time"], unit="s", errors="coerce", utc=True)
quiz_attempts["question_key"] = (
    "quiz_" + quiz_attempts["quiz_id"].astype("string") + "_" + quiz_attempts["question_id"].astype("string")
)

attempts_all = pd.concat(
    [
        math_results[["user_id", "attempt_time", "question_key", "score_ratio"]],
        quiz_attempts[["user_id", "attempt_time", "question_key", "score_ratio"]],
    ],
    ignore_index=True,
).dropna(subset=["user_id", "attempt_time", "score_ratio"])
attempts_all["user_id"] = attempts_all["user_id"].astype("int64")

achievement_base = (
    attempts_all.groupby("user_id")
    .agg(
        average_score_ratio=("score_ratio", "mean"),
        score_variance=("score_ratio", "std"),
    )
    .reset_index()
)

## **Step 1: K-Means on Effort, Content Depth, Practice Review and Help Seeking**

### Merge all required RQ features

In [11]:
# Merge all required RQ features
feature_frames = [
    active_days_spacing[[ 'user_id', 'active_days']],
    total_active_learning[['user_id', 'total_active_learning_time_seconds']],
    video_features[['user_id', 'avg_video_completion_ratio', 'video_completion_rate', 'videos_interacted', 'pause_per_video', 'replay_per_video']],
    scroll_features[['user_id', 'avg_scroll_depth_ratio']],
    chat_features[['user_id', 'chatbot_user_messages_per_thread', 'chatbot_prompt_effort_chars']],
    achievement_base[['user_id', 'average_score_ratio']],
    question_review[['user_id', 'repeated_question_views', 'unique_questions_viewed', 'total_question_views']]
]

features = students.copy()
for frame in feature_frames:
    features = features.merge(frame, on="user_id", how="left")

features = features.sort_values("user_id").reset_index(drop=True)

print("Feature table shape:", features.shape)
features.head()

Feature table shape: (1781, 15)


,user_id,active_days,total_active_learning_time_seconds,avg_video_completion_ratio,video_completion_rate,videos_interacted,pause_per_video,replay_per_video,avg_scroll_depth_ratio,chatbot_user_messages_per_thread,chatbot_prompt_effort_chars,average_score_ratio,repeated_question_views,unique_questions_viewed,total_question_views
0,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
1,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.702128,NaN,NaN,NaN
2,113,8.0,381.411,NaN,NaN,NaN,NaN,NaN,0.532108,NaN,NaN,0.641304,NaN,NaN,NaN
3,116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.608696,NaN,NaN,NaN
4,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.746988,NaN,NaN,NaN


In [12]:
print("Feature table shape before dropping NaNs:", features.shape)
features = features.dropna()
print("Feature table shape after dropping NaNs:", features.shape)

Feature table shape before dropping NaNs: (1781, 15)
Feature table shape after dropping NaNs: (531, 15)


### Initialize results dict and common functions

In [13]:
results: dict[str, dict] = {}

In [14]:
# ── Search grid ───────────────────────────────────────────────────────────────
K_RANGE      = range(2, 9)
RANDOM_STATE = 42
N_INIT       = 20          # K-Means restarts per K

In [15]:
def grid_search(X: np.ndarray) -> dict:
    """
    Grid search over K.
    Picks the K that maximises the Silhouette score.
    """
    best = {"score": -np.inf, "k": None, "labels": None}

    for k in K_RANGE:
        try:
            labels = KMeans(
                n_clusters=k,
                n_init=N_INIT,
                random_state=RANDOM_STATE,
            ).fit_predict(X)

            if len(np.unique(labels)) < 2:      # degenerate — skip
                continue

            score = silhouette_score(X, labels)
            if score > best["score"]:
                best.update(
                    score=score,
                    k=k,
                    labels=labels.copy(),
                )

        except Exception:
            continue

    return best


# ── Per-dimension runner ──────────────────────────────────────────────────────

def cluster_dimension(name: str, feature_cols: list[str], df: pd.DataFrame) -> dict:
    print(f"\n{'─' * 58}")
    print(f"  Dimension : {name}")
    print(f"  Features  : {feature_cols}")
    print(f"{'─' * 58}")

    valid_mask = df[feature_cols].notna().any(axis=1)
    n_dropped  = (~valid_mask).sum()
    if n_dropped:
        print(f"  ⚠  {n_dropped} students dropped (all features NaN).")

    X = preprocess(df[valid_mask], feature_cols)

    best = grid_search(X)

    if best["k"] is None:
        print("  ✗  No valid clustering found.")
    else:
        print(f"  Best K    : {best['k']}")
        print(f"  Silhouette: {best['score']:.4f}")
        for cid, cnt in pd.Series(best["labels"]).value_counts().sort_index().items():
            print(f"    Cluster {cid}: {cnt} students")

    best["valid_mask"] = valid_mask
    return best

def cluster_centres(dim_name, DIMENSIONS, results, features):
    feature_cols = DIMENSIONS[dim_name]
    res          = results[dim_name]

    # Get the preprocessed (but original-scale) data for valid students
    df_valid = features[res["valid_mask"]][feature_cols].copy()
    df_valid["cluster"] = res["labels"]

    # Compute per-cluster means on the RAW features (before scaling)
    centroids = df_valid.groupby("cluster")[feature_cols].mean()
    
    print(f"\n--- Centroids for dimension: {dim_name} ----")
    print(centroids.round(2).to_string())

### **Effort**

In [16]:
dim_name = "Effort"

DIMENSIONS = {
    "Effort": [
        "total_active_learning_time_seconds",
        "active_days",
        
    ],
}

def preprocess(df: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    X = df[feature_cols].copy().astype(float)
    X = SimpleImputer(strategy="median").fit_transform(X)
    X = StandardScaler().fit_transform(X)
    return X

results[dim_name] = cluster_dimension(dim_name, DIMENSIONS[dim_name], features)

cluster_centres(dim_name, DIMENSIONS, results, features)


──────────────────────────────────────────────────────────
  Dimension : Effort
  Features  : ['total_active_learning_time_seconds', 'active_days']
──────────────────────────────────────────────────────────
  Best K    : 2
  Silhouette: 0.5366
    Cluster 0: 379 students
    Cluster 1: 152 students

--- Centroids for dimension: Effort ----
         total_active_learning_time_seconds  active_days
cluster                                                 
0                                  37002.80        36.24
1                                 108676.88        79.89


In [17]:
EFFORT_LABELS = {0: "Low Effort", 1: "High Effort"}
res = results[dim_name]
res['labels'] = [EFFORT_LABELS[label] for label in res["labels"]]

### **Content_Depth**

In [18]:
dim_name = "Content_Depth"

DIMENSIONS = {
    "Content_Depth": [
        "avg_video_completion_ratio",
        "video_completion_rate",
        "videos_interacted",
        "pause_per_video",
        "replay_per_video",
        "avg_scroll_depth_ratio",
    ],
}

def preprocess(df: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    """
    Impute → log1p skewed columns → winsorise → RobustScaler.

    RobustScaler (centres on median, scales by IQR) is inherently less
    sensitive to remaining extremes than StandardScaler.
    """
    X = df[feature_cols].copy().astype(float)

    # 1. Median imputation
    X = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(X),
        columns=feature_cols,
    )

    # 2. Log1p-transform skewed features (values are non-negative by construction)
    for col in feature_cols:
        if col in ['videos_interacted', 'pause_per_video', 'replay_per_video']:  # count-based features can be zero but not negative
            X[col] = np.log1p(X[col].clip(lower=0))

    # 3. Winsorise at 1st–99th percentile (per column)
    for col in feature_cols:
        lo = X[col].quantile(0.01)
        hi = X[col].quantile(0.99)
        X[col] = X[col].clip(lower=lo, upper=hi)

    # 4. Robust scaling
    return RobustScaler().fit_transform(X)

results[dim_name] = cluster_dimension(dim_name, DIMENSIONS[dim_name], features)

cluster_centres(dim_name, DIMENSIONS, results, features)


──────────────────────────────────────────────────────────
  Dimension : Content_Depth
  Features  : ['avg_video_completion_ratio', 'video_completion_rate', 'videos_interacted', 'pause_per_video', 'replay_per_video', 'avg_scroll_depth_ratio']
──────────────────────────────────────────────────────────
  Best K    : 2
  Silhouette: 0.2605
    Cluster 0: 382 students
    Cluster 1: 149 students

--- Centroids for dimension: Content_Depth ----
         avg_video_completion_ratio  video_completion_rate  videos_interacted  pause_per_video  replay_per_video  avg_scroll_depth_ratio
cluster                                                                                                                                 
0                              0.74                   0.62              15.20              3.1              0.87                    0.61
1                              0.37                   0.17               5.03              1.6              0.33                    0.60


In [19]:
CONTENT_LABELS = {0: "Deep Engagers", 1: "Surface Browsers"}
res = results[dim_name]
res['labels'] = [CONTENT_LABELS[label] for label in res["labels"]]

### **Practice_Review**

In [20]:
dim_name = "Practice_Review"

DIMENSIONS = {
    "Practice_Review": [
        "repeated_question_views",
        "unique_questions_viewed",
        "total_question_views",
    ],
}

def preprocess(df: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    """Median-impute then z-score all features in a dimension."""
    X = df[feature_cols].copy().astype(float)

    # 1. Median imputation
    X = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(X),
        columns=feature_cols,
    )
    # 2. Log1p-transform skewed features (values are non-negative by construction)
    for col in feature_cols:
        if col in ['repeated_question_views', 'total_question_views']:  # count-based features can be zero but not negative
            X[col] = np.log1p(X[col].clip(lower=0))

    # 3. Winsorise at 1st–99th percentile (per column)
    for col in feature_cols:
        lo = X[col].quantile(0.01)
        hi = X[col].quantile(0.99)
        X[col] = X[col].clip(lower=lo, upper=hi)

    X = RobustScaler().fit_transform(X)
    return X

results[dim_name] = cluster_dimension(dim_name, DIMENSIONS[dim_name], features)

cluster_centres(dim_name, DIMENSIONS, results, features)


──────────────────────────────────────────────────────────
  Dimension : Practice_Review
  Features  : ['repeated_question_views', 'unique_questions_viewed', 'total_question_views']
──────────────────────────────────────────────────────────
  Best K    : 2
  Silhouette: 0.5290
    Cluster 0: 108 students
    Cluster 1: 423 students

--- Centroids for dimension: Practice_Review ----
         repeated_question_views  unique_questions_viewed  total_question_views
cluster                                                                        
0                          12.94                     8.09                 21.03
1                         134.35                    11.18                145.54


In [21]:
PRACTICE_LABELS = {0: "Light Reviewers", 1: "Deep Reviewers"}
res = results[dim_name]
res['labels'] = [PRACTICE_LABELS[label] for label in res["labels"]]

### **Help_Seeking**

In [22]:
dim_name = "Help_Seeking"

DIMENSIONS = {
    "Help_Seeking": [
        "chatbot_prompt_effort_chars",
        "chatbot_user_messages_per_thread"
    ],
}

def preprocess(df: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    """Median-impute then z-score all features in a dimension."""
    X = df[feature_cols].copy().astype(float)

    # 1. Median imputation
    X = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(X),
        columns=feature_cols,
    )
    # 2. Log1p-transform skewed features (values are non-negative by construction)
    for col in feature_cols:
        if col in ['chatbot_prompt_effort_chars', 'chatbot_user_messages_per_thread']:  # count-based features can be zero but not negative
            X[col] = np.log1p(X[col].clip(lower=0))

    # 3. Winsorise at 1st–99th percentile (per column)
    for col in feature_cols:
        lo = X[col].quantile(0.01)
        hi = X[col].quantile(0.99)
        X[col] = X[col].clip(lower=lo, upper=hi)

    X = RobustScaler().fit_transform(X)
    return X

results[dim_name] = cluster_dimension(dim_name, DIMENSIONS[dim_name], features)

cluster_centres(dim_name, DIMENSIONS, results, features)


──────────────────────────────────────────────────────────
  Dimension : Help_Seeking
  Features  : ['chatbot_prompt_effort_chars', 'chatbot_user_messages_per_thread']
──────────────────────────────────────────────────────────
  Best K    : 2
  Silhouette: 0.3794
    Cluster 0: 147 students
    Cluster 1: 384 students

--- Centroids for dimension: Help_Seeking ----
         chatbot_prompt_effort_chars  chatbot_user_messages_per_thread
cluster                                                               
0                             197.54                              4.18
1                              39.88                              5.68


In [23]:
HELP_LABELS = {0: "Deliberate Askers", 1: "Passive Chatters"}
res = results[dim_name]
res['labels'] = [HELP_LABELS[label] for label in res["labels"]]

## **Step 2: K-Modes on Profiles**

### Prepare labels_df

In [24]:
# ── Assemble label DataFrame (input to K-Modes) ───────────────────────────────
labels_df = features[["user_id"]].copy().reset_index(drop=True)

for dim_name, res in results.items():
    col = f"label_{dim_name}"
    labels_df[col] = ""
    if res["labels"] is not None:
        valid_idx = labels_df.index[res["valid_mask"].values]
        labels_df.loc[valid_idx, col] = res["labels"]

print("\n" + "═" * 58)
print("  Label DataFrame — ready for K-Modes")
print("═" * 58)
print(labels_df.head(10).to_string(index=False))
print(f"\n  Shape : {labels_df.shape}")
print(f"  NaNs  :\n{labels_df.isna().sum().to_string()}")


══════════════════════════════════════════════════════════
  Label DataFrame — ready for K-Modes
══════════════════════════════════════════════════════════
 user_id label_Effort label_Content_Depth label_Practice_Review label_Help_Seeking
     127   Low Effort       Deep Engagers        Deep Reviewers  Deliberate Askers
     525   Low Effort    Surface Browsers        Deep Reviewers  Deliberate Askers
     564   Low Effort    Surface Browsers        Deep Reviewers   Passive Chatters
     568   Low Effort    Surface Browsers        Deep Reviewers   Passive Chatters
     569   Low Effort       Deep Engagers        Deep Reviewers   Passive Chatters
     572   Low Effort    Surface Browsers        Deep Reviewers  Deliberate Askers
     574  High Effort    Surface Browsers        Deep Reviewers   Passive Chatters
     575  High Effort       Deep Engagers        Deep Reviewers  Deliberate Askers
     577   Low Effort    Surface Browsers       Light Reviewers   Passive Chatters
     584   Lo

In [25]:
labels_df.head()

,user_id,label_Effort,label_Content_Depth,label_Practice_Review,label_Help_Seeking
0,127,Low Effort,Deep Engagers,Deep Reviewers,Deliberate Askers
1,525,Low Effort,Surface Browsers,Deep Reviewers,Deliberate Askers
2,564,Low Effort,Surface Browsers,Deep Reviewers,Passive Chatters
3,568,Low Effort,Surface Browsers,Deep Reviewers,Passive Chatters
4,569,Low Effort,Deep Engagers,Deep Reviewers,Passive Chatters


### **Check if average_score_ratio is normally distributed**

In [26]:
score_col = 'average_score_ratio'
scores = features[score_col].dropna()
stat, p = stats.shapiro(scores)

print(f"Shapiro-Wilk Test on '{score_col}'")
print(f"  W = {stat:.4f}")
print(f"  p = {p:.4e}")

if p < 0.05:
    print("  → Not normally distributed (p < 0.05). Use non-parametric tests.")
else:
    print("  → Normally distributed (p ≥ 0.05). Parametric tests are appropriate.")

Shapiro-Wilk Test on 'average_score_ratio'
  W = 0.9674
  p = 1.7942e-09
  → Not normally distributed (p < 0.05). Use non-parametric tests.


### **K-Modes Clustering**

In [27]:
"""
K-Modes Clustering + Profile Scoring via average_score_ratio
=============================================================
Following Mejia-Domenzain et al., average_score_ratio is held out as the
outcome variable — not used during clustering. After profiling, we compute mean average_score_ratio per profile

Assumes in scope:
  - labels_df  : output of spectral clustering (user_id + label_* columns)
  - features   : original feature DataFrame containing user_id + average_score_ratio
"""

# ── 1. Prepare label matrix ───────────────────────────────────────────────────

label_cols  = [c for c in labels_df.columns if c.startswith("label_")]
valid_cols  = [c for c in label_cols if labels_df[c].notna().any()]
dropped     = set(label_cols) - set(valid_cols)
if dropped:
    print(f"⚠  Dropped all-NaN dimension(s): {dropped}")

df_km = labels_df[["user_id"] + valid_cols].dropna(subset=valid_cols).copy()

X_cat = df_km[valid_cols].values
print(f"   Students : {len(df_km)}   |   Dimensions : {len(valid_cols)}\n")


# ── 2. Hamming-distance Silhouette ────────────────────────────────────────────

def hamming_silhouette(X: np.ndarray, labels: np.ndarray) -> float:
    X_enc = np.zeros_like(X, dtype=float)
    for j in range(X.shape[1]):
        le = LabelEncoder()
        X_enc[:, j] = le.fit_transform(X[:, j])
    return silhouette_score(X_enc, labels, metric="hamming")


# ── 3. Grid-search K ─────────────────────────────────────────────────────────

K_RANGE      = range(2, 9)
N_INIT       = 10
RANDOM_STATE = 42

print("  K   Silhouette   Cost")
print("  " + "─" * 26)

search_results = []
for k in K_RANGE:
    km     = KModes(n_clusters=k, init="Huang", n_init=N_INIT,
                    random_state=RANDOM_STATE, verbose=0)
    lbls   = km.fit_predict(X_cat)
    if len(np.unique(lbls)) < 2:
        continue
    sil = hamming_silhouette(X_cat, lbls)
    search_results.append({"k": k, "silhouette": sil, "cost": km.cost_,
                            "labels": lbls.copy(), "model": km})
    print(f"  {k}   {sil:>9.4f}   {km.cost_:.1f}")

best = max(search_results, key=lambda r: r["silhouette"])
print(f"\n  ✔  Best K = {best['k']}  (Silhouette = {best['silhouette']:.4f})\n")

df_km["profile"] = best["labels"]


# ── 4. Merge average_score_ratio ──────────────────────────────────────────────

df_scored = df_km.merge(
    features[["user_id", score_col]].dropna(subset=[score_col]),
    on="user_id",
    how="inner",
)

   Students : 531   |   Dimensions : 4

  K   Silhouette   Cost
  ──────────────────────────
  2      0.4880   404.0
  3      0.5532   280.0
  4      0.5702   221.0
  5      0.6010   181.0
  6      0.6640   140.0
  7      0.7761   82.0
  8      0.8125   63.0

  ✔  Best K = 8  (Silhouette = 0.8125)



In [28]:
# ── 5. Per-profile score statistics ───────────────────────────────────────────

agg = (
    df_scored.groupby("profile")[score_col]
    .agg(Number_of_Students="count", Mean_Score="mean", Std_Score="std")
    .round(4)
    .sort_values("Mean_Score", ascending=False)
)

# Rank profiles by median score (1 = best)
agg["Score_Rank"] = range(1, len(agg) + 1)

# Append dimension mode for each profile
for col in valid_cols:
    dim_name = col.replace("label_", "")
    agg[dim_name] = [
        df_km.loc[df_km["profile"] == pid, col].mode()[0]
        for pid in agg.index
    ]

print("═" * 80)
print("  Profile Score Summary  (sorted by mean average_score_ratio, best first)")
print("═" * 80)
print(agg.to_string())

════════════════════════════════════════════════════════════════════════════════
  Profile Score Summary  (sorted by mean average_score_ratio, best first)
════════════════════════════════════════════════════════════════════════════════
         Number_of_Students  Mean_Score  Std_Score  Score_Rank       Effort     Content_Depth  Practice_Review       Help_Seeking
profile                                                                                                                          
4                        55      0.6999     0.1393           1  High Effort     Deep Engagers   Deep Reviewers  Deliberate Askers
5                        40      0.6903     0.1397           2   Low Effort     Deep Engagers   Deep Reviewers  Deliberate Askers
2                        74      0.6784     0.1530           3  High Effort     Deep Engagers   Deep Reviewers   Passive Chatters
7                       156      0.6562     0.1441           4   Low Effort     Deep Engagers   Deep Reviewers   P

### **Run a Kruskal-Wallis test (non-parametric, robust to unequal variance) to check whether profiles differ significantly in score.**

In [29]:
groups = [
    df_scored.loc[df_scored["profile"] == pid, score_col].values
    for pid in sorted(df_scored["profile"].unique())
]
kw_stat, kw_p = stats.kruskal(*groups)

print(f"\n  Kruskal-Wallis  H = {kw_stat:.3f},  p = {kw_p:.4e}")
if kw_p < 0.05:
    print("  → Profiles differ significantly in average_score_ratio (p < 0.05).")
else:
    print("  → No significant overall difference (p ≥ 0.05).\n")


  Kruskal-Wallis  H = 18.963,  p = 8.3037e-03
  → Profiles differ significantly in average_score_ratio (p < 0.05).
